# Parte VI. Reporte técnico interactivo

**Integrantes:** Ricardo Amiel Acuña Villogas, Juan Leibniz Aquino Espinoza y Camilo Ernesto Soto Cristóbal.

Este notebook arma la página web del proyecto. No calcula nada: toma los agregados que exportaron los notebooks 1 a 5, los incrusta en la plantilla de **reporte/plantilla.html** y escribe **reporte/index.html**.

La arquitectura es la que recomienda el enunciado, calcular una vez y explorar muchas. Spark hace el trabajo pesado fuera de línea y deja tablas pequeñas; la página lee esas tablas ya incrustadas y responde al instante sin invocar Spark. La recomputación en vivo se reserva para lo barato, que son las fórmulas cerradas de los algoritmos aproximados.

In [1]:
import sys, time
from pathlib import Path

# Los ayudantes del proyecto viven en notebooks/scripts. Se importan por
# nombre, no con asterisco, para que quede claro de donde sale cada uno.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "notebooks" / "scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.helpers import (PARQUET, FIGS, ARTEFACTOS, FIGURAS, PALETA,
                     AZUL, ROJO, VERDE, MORADO, TEAL, TINTA, SUAVE, fmt,
                     abrir_spark, cronometro, medir, figura, exportar,
                     barras_h, lineas, barras_agrupadas, mapa_calor, lorenz, gini)

pd.set_option("display.width", 190)
pd.set_option("display.max_colwidth", 64)
import json

PLANTILLA = RAIZ / "reporte" / "plantilla.html"
SALIDA = RAIZ / "reporte" / "index.html"
print(f"plantilla: {PLANTILLA.stat().st_size/1024:.0f} KiB")

plantilla: 87 KiB


## 1. Reunir los agregados

Cada entrada del diccionario es una tabla que algún notebook anterior exportó. Si falta alguna, la celda lo dice por su nombre en lugar de fallar con un error críptico.

In [2]:
def leer(nombre):
    """Lee un artefacto exportado por los notebooks 1 a 5."""
    ruta = ARTEFACTOS / f"{nombre}.json"
    if not ruta.exists():
        raise FileNotFoundError(
            f"falta {nombre}.json; hay que ejecutar antes el notebook que lo produce")
    return json.loads(ruta.read_text(encoding="utf-8"))


DATOS = {
    "portada":             leer("p1_portada")[0],
    "serie_mensual":       leer("p1_serie_mensual"),
    "concentracion":       leer("p1_concentracion"),
    "gini_departamento":   leer("p1_concentracion_departamento"),
    "competencia":         leer("p1_competencia"),
    "estacionalidad":      leer("p1_estacionalidad"),
    "lorenz":              leer("p1_lorenz"),
    "histograma":          leer("p1_histograma_monto"),
    "error_minhash":       leer("p2_error_minhash"),
    "barrido_lsh":         leer("p2_barrido_lsh"),
    "casos":               leer("p2_casos"),
    "ann":                 leer("p3_comparacion"),
    "escalado":            leer("p3_escalado"),
    "bloom":               leer("p4_bloom"),
    "cms":                 leer("p4_cms"),
    "dgim":                leer("p4_dgim"),
    "resumen_estructuras": leer("p4_resumen"),
    "resumen_flujos":      leer("p4_escalares")[0],
    "costo_algoritmos":    leer("p5_costo_algoritmos"),
    "reglas":              leer("p5_reglas"),
}

resumen = pd.DataFrame([{"bloque": k,
                         "filas": len(v) if isinstance(v, list) else 1,
                         "campos": len(v[0]) if isinstance(v, list) and v else len(v)}
                        for k, v in DATOS.items()])
print(f"{len(DATOS)} bloques reunidos")
resumen

20 bloques reunidos


,bloque,filas,campos
0,portada,1,4
1,serie_mensual,36,3
2,concentracion,4,3
3,gini_departamento,25,4
4,competencia,6,4
5,estacionalidad,12,4
6,lorenz,200,2
7,histograma,43,3
8,error_minhash,4,4
9,barrido_lsh,7,11


## 2. Incrustar y escribir la página

Los datos van dentro del archivo y no se cargan por separado. El motivo es práctico: una página abierta desde el disco con el esquema de archivo local no puede pedir un JSON vecino, porque el navegador lo bloquea por seguridad. Incrustarlos hace que el archivo funcione con un doble clic, sin levantar un servidor, que es lo que se necesita para entregarlo.

In [3]:
plantilla = PLANTILLA.read_text(encoding="utf-8")
if "__DATOS__" not in plantilla:
    raise ValueError("la plantilla no tiene el marcador __DATOS__")

carga = json.dumps(DATOS, ensure_ascii=False, separators=(",", ":"))
html = plantilla.replace("__DATOS__", carga)

SALIDA.parent.mkdir(parents=True, exist_ok=True)
SALIDA.write_text(html, encoding="utf-8")

# Segunda copia en docs. GitHub Pages solo admite la raiz del repositorio o la
# carpeta docs, nunca una carpeta arbitraria, asi que el reporte se publica
# desde ahi. Se escribe desde el mismo paso para que las dos no se desincronicen.
PAGES = RAIZ / "docs" / "index.html"
PAGES.parent.mkdir(parents=True, exist_ok=True)
PAGES.write_text(html, encoding="utf-8")

print(f"datos incrustados: {len(carga)/1024:.0f} KiB")
print(f"pagina escrita:    {SALIDA.stat().st_size/1024:.0f} KiB  ->  {SALIDA}")
print(f"copia para Pages:  {PAGES}")

datos incrustados: 71 KiB
pagina escrita:    157 KiB  ->  /home/germain/Documentos/Python_Projects/DataMining/PROYECTO/reporte/index.html
copia para Pages:  /home/germain/Documentos/Python_Projects/DataMining/PROYECTO/docs/index.html


## 3. Verificación

Tres comprobaciones antes de dar la página por buena: que no quedó ningún marcador sin reemplazar, que los datos incrustados vuelven a leerse como JSON válido, y que cada identificador que el script busca en el documento existe de verdad en el HTML.

In [4]:
import re

problemas = []
if "__DATOS__" in html:
    problemas.append("quedó el marcador sin reemplazar")

incrustado = re.search(r"const D = (\{.*?\});\n", html, re.S)
try:
    recuperado = json.loads(incrustado.group(1))
    if set(recuperado) != set(DATOS):
        problemas.append("los bloques incrustados no coinciden con los reunidos")
except Exception as e:
    problemas.append(f"los datos incrustados no son JSON válido: {e}")

buscados = set(re.findall(r'getElementById\("([^"]+)"\)', html))
presentes = set(re.findall(r'id="([^"]+)"', html))
faltantes = sorted(buscados - presentes)
if faltantes:
    problemas.append(f"el script busca identificadores que no existen: {faltantes}")

print(f"bloques verificados: {len(recuperado)}")
print(f"identificadores usados por el script: {len(buscados)}, todos presentes: {not faltantes}")
print("\n" + ("verificación: todo en orden" if not problemas else "PROBLEMAS:\n  " + "\n  ".join(problemas)))

bloques verificados: 20
identificadores usados por el script: 33, todos presentes: True

verificación: todo en orden


## 4. Cómo se ve

La página se abre con un doble clic sobre reporte/index.html, sin servidor ni conexión a Spark. Necesita acceso a internet solo para las tipografías y para la biblioteca D3, ambas desde redes de distribución públicas.

Estructura de cada hallazgo, igual en los seis bloques: pregunta de interés, arquitectura del pipeline, visualización, controles donde apliquen, e interpretación escrita. Los cuatro módulos con parámetros ajustables en vivo son la curva S de LSH según bandas y filas, la tasa de falsos positivos del filtro de Bloom según bits por elemento y número de hashes, la cota de error del Count-Min Sketch según ancho y profundidad, y el filtrado de reglas por soporte, confianza y lift. Los tres primeros se recalculan con la fórmula cerrada del algoritmo, que es instantánea; el cuarto filtra sobre las reglas ya calculadas.

In [5]:
from IPython.display import IFrame, display
print(f"Abrir: {SALIDA}")
display(IFrame(src="../reporte/index.html", width="100%", height=620))

Abrir: /home/germain/Documentos/Python_Projects/DataMining/PROYECTO/reporte/index.html
